# Day 4 lab: quantization

**Goal.** You quantize tensors by hand, you quantize a small CNN after the
training and with quantization-aware training, and you make the float model
and the `int8` model that you compare on the board.

**Deliverable.** Four files in `sketches/motion_quant/`: `model_float.h`,
`model_int8.h`, `model_settings.h`, and `test_set.h`. The numbers of this
notebook go into `report.md`.

**Time.** Part A: 30 min, sections 0 to 4. Part B: 50 min, sections 5 to 9.
Part C: 40 min, sections 10 to 13. Part D: about 5 min of its 30 min,
sections 14 and 15. The complete notebook runs in about one minute.

**How to work.** Run the cells in order. A cell with the mark
`TODO (student)` has a task. The notebook has eight tasks. The cell after a
task checks your result.

## 0. Setup

Run this cell first. It prints the versions that you write in your report,
and it selects the dataset.

- If the folder `Labs/day02/data/` has recordings, the notebook uses them.
- If not, the notebook uses the fallback dataset of Day 2 in
  `Labs/day02/fallback_data/`. The fallback signals are simulated. Nobody
  recorded them. A model that learns from them does not work on the real
  kit.

In [ ]:
import contextlib
import io
import json
import os
import platform
import sys
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # no start messages of TensorFlow

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter
from keras import ops

tf.get_logger().setLevel("ERROR")            # no INFO lines of the converter
warnings.filterwarnings("ignore", message="Statistics for quantized inputs")

print("python    ", sys.version.split()[0])
print("numpy     ", np.__version__)
print("tensorflow", tf.__version__)
print("keras     ", keras.__version__)
print("machine   ", platform.machine())

RATE_HZ = 50
WINDOW = 100          # samples: 2 s at 50 Hz
STRIDE = 10           # samples: 0.2 s
FFT_LENGTH = 32
CLASSES = ["idle", "terrestrial", "lift", "maritime"]

# The lab folder. The notebook of the folder solutions/ uses the folder above.
LAB = "." if os.path.isdir("sketches") and os.path.isdir("host") else ".."
DAY2 = os.path.join(LAB, "..", "day02")
SKETCH = os.path.join(LAB, "sketches", "motion_quant")
sys.path.insert(0, os.path.join(LAB, "host"))


def recording_files(folder):
    '''Return the names of the recording files in a folder.'''
    if not os.path.isdir(folder):
        return []
    return sorted(name for name in os.listdir(folder)
                  if name.endswith(".csv") and len(name.split(".")) == 4)


DATA = os.path.join(DAY2, "data")
if not recording_files(DATA):
    DATA = os.path.join(DAY2, "fallback_data")
    if not recording_files(DATA):
        sys.path.insert(0, os.path.join(DAY2, "host"))
        import make_fallback_dataset
        import motion_sim
        os.makedirs(DATA, exist_ok=True)
        for label in motion_sim.CLASSES:
            for session in make_fallback_dataset.SESSIONS:
                for number in range(1, make_fallback_dataset.RECORDINGS + 1):
                    make_fallback_dataset.write_recording(DATA, label, session,
                                                          number)
    print()
    print("No recording in Labs/day02/data/. The notebook uses the fallback")
    print("dataset. Its signals are simulated. Nobody recorded them.")
print()
print("data folder:", os.path.normpath(DATA), "with",
      len(recording_files(DATA)), "files")


@contextlib.contextmanager
def no_status_messages():
    '''Hide the status lines that TensorFlow prints during a conversion.'''
    sys.stdout.flush()
    sys.stderr.flush()
    saved = os.dup(1), os.dup(2)
    with open(os.devnull, "w") as null, contextlib.redirect_stdout(io.StringIO()):
        os.dup2(null.fileno(), 1)
        os.dup2(null.fileno(), 2)
        try:
            yield
        finally:
            os.dup2(saved[0], 1)
            os.dup2(saved[1], 2)
            os.close(saved[0])
            os.close(saved[1])


def convert_quietly(converter):
    '''Call converter.convert() and hide its status lines.'''
    with no_status_messages():
        return converter.convert()


def make_interpreter(tflite_model):
    '''Return a LiteRT interpreter for the model, with its tensors ready.'''
    with no_status_messages():
        interpreter = Interpreter(model_content=tflite_model)
        interpreter.allocate_tensors()
    return interpreter

## 1. The dataset (2 min)

The code loads the recordings of Day 2 with the split of Day 2, as the
notebook of Day 3 does. One window has 100 samples of the three acceleration
axes in $g$.

You do not change this code.

In [ ]:
def load_recording(folder, name):
    '''Return one recording: label, session, and the samples of ax, ay, az.'''
    label, session, _, _ = name.split(".")
    rows = []
    with open(os.path.join(folder, name), encoding="ascii",
              errors="replace") as handle:
        for line in handle:
            fields = line.strip().split(",")
            if line.startswith("#") or len(fields) != 8 or fields[0] == "n":
                continue
            rows.append([float(f) for f in fields[2:5]])
    return {"name": name, "label": label, "session": session,
            "data": np.array(rows, dtype=np.float32)}


def cut_windows(record):
    '''Return the windows of one recording: array (windows, 100, 3).'''
    data = record["data"]
    starts = range(0, len(data) - WINDOW + 1, STRIDE)
    return np.stack([data[s:s + WINDOW] for s in starts])


records = [load_recording(DATA, name) for name in recording_files(DATA)]
labels = [c for c in CLASSES if any(r["label"] == c for r in records)]
labels += sorted({r["label"] for r in records} - set(labels))
sessions = sorted({r["session"] for r in records})

split_file = os.path.join(DATA, "split.json")
if os.path.exists(split_file):
    with open(split_file, encoding="utf-8") as handle:
        TEST_SESSIONS = json.load(handle)["test_sessions"]
    print("split.json: test sessions", TEST_SESSIONS)
else:
    TEST_SESSIONS = [sessions[-1]]
    print("no split.json: the last session is the test session", TEST_SESSIONS)


def windows_of(session_list):
    '''Return (windows, class numbers) of all recordings of the sessions.'''
    group = [r for r in records if r["session"] in session_list]
    x = np.concatenate([cut_windows(r) for r in group])
    y = np.concatenate([[labels.index(r["label"])] * len(cut_windows(r))
                        for r in group])
    return x, y


train_sessions = [s for s in sessions if s not in TEST_SESSIONS]
w_train, y_train = windows_of(train_sessions)
w_test, y_test = windows_of(TEST_SESSIONS)

print(f"classes:  {labels}")
print(f"training: sessions {train_sessions}, {len(w_train)} windows")
print(f"test:     sessions {TEST_SESSIONS}, {len(w_test)} windows")
print()
print(f"{'class':<14}{'smallest value':>16}{'largest value':>16}")
for number, label in enumerate(labels):
    values = w_train[y_train == number]
    print(f"{label:<14}{values.min():>16.2f}{values.max():>16.2f}")

# Part A: quantize by hand (30 min)

## 2. The affine mapping (10 min)

Part 1 of the lecture gave the mapping between a real value $r$ and an
integer $q$ of the type `int8`:

$$r = S\,(q - Z) \qquad q = \mathrm{clamp}\!\left(\mathrm{round}\!\left(\frac{r}{S}\right) + Z,\ -128,\ 127\right)$$

The scale $S$ and the zero point $Z$ come from the range
$[r_{\min}, r_{\max}]$ of the tensor:

$$S = \frac{r_{\max} - r_{\min}}{255} \qquad Z = \mathrm{round}\!\left(-128 - \frac{r_{\min}}{S}\right)$$

**Task 1.** Write the two lines of the function `scale_zero_point`.

In [ ]:
def scale_zero_point(r_min, r_max):
    '''Return (S, Z) of the int8 mapping for the range [r_min, r_max].'''
    # The range must hold the real value 0. The converter does the same.
    r_min, r_max = min(r_min, 0.0), max(r_max, 0.0)
    # TODO (student): replace the two lines below. Z is a Python int.
    S = 1.0
    Z = 0
    return S, Z

**Task 2.** Write the functions `quantize` and `dequantize`. Each function
needs one line. Use `np.round` and `np.clip`.

In [ ]:
def quantize(r, S, Z):
    '''Return the int8 values of the real values r.'''
    r = np.asarray(r, dtype=np.float64)
    # TODO (student): replace the line below.
    q = np.zeros(r.shape)
    return q.astype(np.int8)


def dequantize(q, S, Z):
    '''Return the real values of the int8 values q.'''
    q = np.asarray(q, dtype=np.float64)
    # TODO (student): replace the line below.
    r = np.zeros(q.shape)
    return r

In [ ]:
def check_tasks_1_and_2():
    '''Compare the three functions with the examples of the lecture.'''
    S, Z = scale_zero_point(-1.0, 3.0)
    task_1 = abs(S - 4.0 / 255) < 1e-9 and Z == -64
    S2, Z2 = scale_zero_point(-1.00, 1.55)
    task_1 = task_1 and abs(S2 - 0.01) < 1e-9 and Z2 == -28
    q = quantize([0.7], 4.0 / 255, -64)
    r = dequantize(q, 4.0 / 255, -64)
    task_2 = int(q[0]) == -19 and abs(float(r[0]) - 0.70588) < 1e-4
    q5 = quantize([0.000, 0.500, -0.333, 1.234, 2.000], 0.01, -28)
    task_2 = task_2 and [int(v) for v in q5] == [-28, 22, -61, 95, 127]
    print("Task 1:", "complete" if task_1 else "not complete")
    print("Task 2:", "complete" if task_2 else "not complete")
    return task_1, task_2


task_1_complete, task_2_complete = check_tasks_1_and_2()

# The exercise of Part 1 of the lecture, with your functions.
S, Z = scale_zero_point(-1.00, 1.55)
print()
print(f"range [-1.00, 1.55]: S = {S:.6f}, Z = {Z}")
print(f"{'r':>8}{'q':>6}{'r_hat':>9}{'error':>9}")
for r in (0.000, 0.500, -0.333, 1.234, 2.000):
    q = quantize([r], S, Z)[0]
    r_hat = dequantize([q], S, Z)[0]
    print(f"{r:>8.3f}{int(q):>6}{r_hat:>9.3f}{r_hat - r:>9.3f}")

## 3. The rounding error of a real tensor (6 min)

The tensor of this section is the input of the model: all acceleration values
of the training windows. The code quantizes the tensor with your functions
and measures the error $\hat{r} - r$.

The lecture gave two rules for a value inside the range:

- the largest error is $S/2$,
- the typical error (RMS) is $S/\sqrt{12}$, about $0.29\,S$.

Check the two rules with the output. Write the four numbers in `report.md`.

In [ ]:
tensor = w_train.reshape(-1).astype(np.float64)
r_min, r_max = float(tensor.min()), float(tensor.max())
S, Z = scale_zero_point(r_min, r_max)
error = dequantize(quantize(tensor, S, Z), S, Z) - tensor

print(f"tensor: {tensor.size} values from {r_min:.4f} to {r_max:.4f}")
print(f"scale S = {S:.6f}, zero point Z = {Z}")
print(f"largest error:        {np.abs(error).max():.6f}   (S / 2 = {S / 2:.6f})")
print(f"typical error (RMS):  {np.sqrt(np.mean(error ** 2)):.6f}   "
      f"(S / sqrt(12) = {S / np.sqrt(12):.6f})")
print(f"bytes: {tensor.size * 4} in float32, {tensor.size} in int8")

The next cell uses the same range with fewer bits. With $b$ bits, the range
has $2^b - 1$ steps. The function `quantize_with_bits` returns the real
values after the two steps: quantize, then dequantize.

Before you run the cell: by which factor does the error grow when you remove
one bit?

In [ ]:
def quantize_with_bits(r, r_min, r_max, bits):
    '''Quantize r with 2**bits levels, then dequantize. Return r_hat.'''
    r_min, r_max = min(r_min, 0.0), max(r_max, 0.0)
    levels = 2 ** bits - 1
    S = (r_max - r_min) / levels
    Z = round(-r_min / S)
    q = np.clip(np.round(r / S) + Z, 0, levels)
    return S * (q - Z)


def rms(values):
    '''Return the root of the mean of the squares.'''
    return float(np.sqrt(np.mean(np.square(values))))


print(f"{'bits':>5}{'levels':>8}{'scale':>12}{'RMS error':>12}{'against 8 bits':>16}")
rms_8 = rms(quantize_with_bits(tensor, r_min, r_max, 8) - tensor)
for bits in (8, 7, 6, 4, 2):
    r_hat = quantize_with_bits(tensor, r_min, r_max, bits)
    scale = (max(r_max, 0.0) - min(r_min, 0.0)) / (2 ** bits - 1)
    print(f"{bits:>5}{2 ** bits:>8}{scale:>12.6f}{rms(r_hat - tensor):>12.6f}"
          f"{rms(r_hat - tensor) / rms_8:>15.1f}x")

# One window of the last class, axis z: the float signal and two bit widths.
index = int(np.argmax(y_train == y_train.max()))
signal = w_train[index, :, 2].astype(np.float64)
time_s = np.arange(WINDOW) / RATE_HZ
figure, axis = plt.subplots(figsize=(8.5, 2.8))
axis.plot(time_s, signal, color="black", linewidth=1.0, label="float32")
for bits, colour in ((4, "tab:red"), (2, "tab:blue")):
    axis.step(time_s, quantize_with_bits(signal, r_min, r_max, bits),
              where="mid", color=colour, linewidth=1.0, label=f"{bits} bits")
axis.set_xlabel("time (s)")
axis.set_ylabel("acceleration z (g)")
axis.set_title(f"one window of the class '{labels[y_train[index]]}'", fontsize=9)
axis.legend(fontsize=8)
axis.grid(alpha=0.3)
figure.tight_layout()
plt.show()

## 4. The clipping error (12 min)

The range decides the scale. A smaller range gives a smaller step, but it
cuts the values outside the range. The next experiment compares four ranges
for the same tensor:

| Range | Content |
|---|---|
| minimum and maximum | all values |
| percentile 99.9 | the range that holds 99.9 percent of the values |
| percentile 99 | the range that holds 99 percent of the values |
| percentile 90 | the range that holds 90 percent of the values |

The experiment runs with 8 bits and with 4 bits.

**Task 3.** Write your prediction **before** you run the experiment. The
prediction is part of the assessment. A wrong prediction costs no points. A
missing prediction does.

In [ ]:
# TODO (student): replace each "?".
prediction_ranges = {
    # "minimum and maximum", "percentile 99.9", "percentile 99", "percentile 90"
    "the range with the smallest RMS error, 8 bits": "?",
    "the range with the smallest RMS error, 4 bits": "?",
    # one sentence
    "the reason": "?",
}
for question, answer in prediction_ranges.items():
    print(f"{question:<48} {answer}")

In [ ]:
def range_of(values, percent):
    '''Return the range that holds this percent of the values.'''
    if percent >= 100:
        return float(values.min()), float(values.max())
    tail = (100 - percent) / 2
    return (float(np.percentile(values, tail)),
            float(np.percentile(values, 100 - tail)))


RANGES = (("minimum and maximum", 100), ("percentile 99.9", 99.9),
          ("percentile 99", 99), ("percentile 90", 90))

for bits in (8, 4):
    print(f"{bits} bits")
    print(f"  {'range':<21}{'from':>8}{'to':>8}{'scale':>10}{'clipped':>9}"
          f"{'RMS error':>11}{'largest':>9}")
    best = None
    for name, percent in RANGES:
        low, high = range_of(tensor, percent)
        r_hat = quantize_with_bits(tensor, low, high, bits)
        error = r_hat - tensor
        clipped = 100 * np.mean((tensor < min(low, 0.0)) | (tensor > max(high, 0.0)))
        scale = (max(high, 0.0) - min(low, 0.0)) / (2 ** bits - 1)
        print(f"  {name:<21}{low:>8.3f}{high:>8.3f}{scale:>10.5f}"
              f"{clipped:>8.2f}%{rms(error):>11.5f}{np.abs(error).max():>9.4f}")
        if best is None or rms(error) < best[1]:
            best = (name, rms(error))
    print(f"  smallest RMS error: {best[0]}")
    print()

> TODO (student): answer these two questions in `report.md`.
>
> 1. Compare the result with your prediction of task 3.
> 2. The converter of LiteRT uses the minimum and the maximum. For which
>    tensor of a model is this a risk? Use the column "scale" of your table.

# Part B: post-training quantization (50 min)

## 5. The float baseline (5 min)

The model of this part is a small CNN for the raw window. It has two
convolutions along the time axis, a global average pooling, and one fully
connected layer.

| Layer | Output | Parameters |
|---|---|---|
| Input | 100 x 3 | none |
| Convolution, 8 filters of 5 samples, stride 2, ReLU | 48 x 8 | 128 |
| Convolution, 16 filters of 5 samples, stride 2, ReLU | 22 x 16 | 656 |
| Global average pooling | 16 | none |
| Fully connected, softmax | 4 | 68 |

The input is the acceleration in $g$, with no normalization. The range of
the input tensor then depends on the motion. Section 9 uses this fact.

All groups train the model with the same seed, so all groups with the same
dataset have the same baseline. You do not change this code.

In [ ]:
def build_cnn():
    '''Return the small CNN for a window of 100 samples and 3 axes.'''
    return keras.Sequential([
        keras.Input(shape=(WINDOW, 3), name="window"),
        keras.layers.Conv1D(8, 5, strides=2, activation="relu", name="conv_1"),
        keras.layers.Conv1D(16, 5, strides=2, activation="relu", name="conv_2"),
        keras.layers.GlobalAveragePooling1D(name="pooling"),
        keras.layers.Dense(len(labels), activation="softmax", name="output"),
    ], name="cnn")


def compile_model(model, learning_rate):
    '''Set the optimizer, the loss, and the metric of a model.'''
    model.compile(optimizer=keras.optimizers.Adam(learning_rate),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])


def class_accuracy(predicted, real):
    '''Return (accuracy, list with the accuracy of each class), in percent.'''
    each = [100 * float(np.mean(predicted[real == k] == k)) if np.any(real == k)
            else float("nan") for k in range(len(labels))]
    return 100 * float(np.mean(predicted == real)), each


keras.utils.set_random_seed(0)
cnn = build_cnn()
compile_model(cnn, 0.005)
cnn.fit(w_train, y_train, epochs=20, batch_size=32, verbose=0)

float_accuracy, float_each = class_accuracy(
    cnn.predict(w_test, verbose=0).argmax(axis=1), y_test)
cnn_float_tflite = convert_quietly(tf.lite.TFLiteConverter.from_keras_model(cnn))

print(f"parameters:              {cnn.count_params()}")
print(f"weights in float32:      {cnn.count_params() * 4} bytes")
print(f"LiteRT file, float32:    {len(cnn_float_tflite)} bytes")
print(f"test accuracy, float32:  {float_accuracy:.1f} percent")
print("for each class:          "
      + ", ".join(f"{label} {value:.1f}" for label, value in zip(labels, float_each)))

## 6. Convert with full integer quantization (10 min)

Part 2 of the lecture showed the settings of the converter. The converter
needs a calibration set: it runs the float model with these inputs and
records the range of each activation.

**Task 4.** Complete the function `convert_int8`:

1. The generator `representative_dataset` gives one input after the other.
   Each item is a list with one array of the type `float32`. The array has
   the batch size 1: for the CNN, the shape is `(1, 100, 3)`.
2. Add the five settings of full integer quantization: `optimizations`,
   `representative_dataset`, `target_spec.supported_ops`,
   `inference_input_type`, and `inference_output_type`.

In [ ]:
def convert_int8(model, calibration):
    '''Return the full integer LiteRT model. calibration: array of inputs.'''
    calibration = np.asarray(calibration, dtype=np.float32)

    def representative_dataset():
        # TODO (student): replace the two lines below. Use a loop over the
        # inputs and the statement "yield".
        return
        yield

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    # TODO (student): add the five settings here.
    return convert_quietly(converter)       # calls converter.convert()

The calibration set has the same number of windows for each class. The
windows come from the training set, not from the test set.

In [ ]:
def calibration_set(x, y, each, classes=None, seed=0):
    '''Return "each" inputs of each class of the list, in a random order.'''
    generator = np.random.default_rng(seed)
    classes = range(len(labels)) if classes is None else classes
    picked = []
    for k in classes:
        positions = np.flatnonzero(y == k)
        picked.extend(generator.permutation(positions)[:each])
    return x[generator.permutation(picked)]


def input_details_of(tflite_model):
    '''Return the description of the input tensor of a LiteRT model.'''
    return make_interpreter(tflite_model).get_input_details()[0]


calibration = calibration_set(w_train, y_train, each=50)
print(f"calibration set: {len(calibration)} windows, values from "
      f"{calibration.min():.4f} to {calibration.max():.4f}")

cnn_int8_tflite = convert_int8(cnn, calibration)
task_4_complete = (len(cnn_int8_tflite) > 0
                   and input_details_of(cnn_int8_tflite)["dtype"] == np.int8)
print("Task 4:", "complete" if task_4_complete else "not complete")
if task_4_complete:
    print()
    print(f"LiteRT file, float32: {len(cnn_float_tflite)} bytes")
    print(f"LiteRT file, int8:    {len(cnn_int8_tflite)} bytes")
    print(f"weights:              {cnn.count_params() * 4} bytes in float32, "
          f"about {cnn.count_params()} bytes in int8")

## 7. Test the `int8` model on the laptop (10 min)

The `int8` model takes `int8` input. The test code must quantize each window
with the scale and the zero point of the input tensor, as your sketch does on
the board.

**Task 5.** Complete the function `quantize_input`. It returns two results:

- the `int8` values. Use your function `quantize`.
- the number of clipped values: the values for which
  $\mathrm{round}(x / S) + Z$ is below $-128$ or above $127$.

In [ ]:
def quantize_input(x, S, Z):
    '''Return (int8 values of x, number of clipped values).'''
    x = np.asarray(x, dtype=np.float64)
    # TODO (student): replace the two lines below.
    q = np.zeros(x.shape, dtype=np.int8)
    clipped = 0
    return q, clipped

In [ ]:
def check_task_5():
    '''Compare quantize_input with the exercise of the lecture.'''
    q, clipped = quantize_input([0.000, 0.500, -0.333, 1.234, 2.000, -1.5],
                                0.01, -28)
    correct = ([int(v) for v in np.ravel(q)] == [-28, 22, -61, 95, 127, -128]
               and int(clipped) == 2)
    print("Task 5:", "complete" if correct else "not complete")
    return correct


def run_litert(tflite_model, x):
    '''Run a LiteRT model for each input of x.

    Return (real outputs, percent of clipped input values, input details).
    '''
    interpreter = make_interpreter(tflite_model)
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    outputs, clipped = [], 0
    for row in x:
        if inp["dtype"] == np.int8:
            S, Z = inp["quantization"]
            row, cut = quantize_input(row, S, Z)
            clipped += cut
        interpreter.set_tensor(inp["index"], row[None, ...].astype(inp["dtype"]))
        interpreter.invoke()
        result = interpreter.get_tensor(out["index"])[0]
        if out["dtype"] == np.int8:
            S_out, Z_out = out["quantization"]
            result = S_out * (result.astype(np.float64) - Z_out)
        outputs.append(result)
    return np.array(outputs), 100 * clipped / x.size, inp


def test_report(name, tflite_model, x, y):
    '''Print one line with the results of a LiteRT model. Return the outputs.'''
    outputs, clipped, _ = run_litert(tflite_model, x)
    accuracy, each = class_accuracy(outputs.argmax(axis=1), y)
    print(f"{name:<22}{len(tflite_model):>7}{accuracy:>10.1f}"
          + "".join(f"{value:>8.1f}" for value in each) + f"{clipped:>9.2f}")
    return outputs


def report_header():
    '''Print the header of the table of test_report.'''
    print(f"{'model':<22}{'bytes':>7}{'accuracy':>10}"
          + "".join(f"{label[:7]:>8}" for label in labels) + f"{'clipped':>9}")


task_5_complete = check_task_5()
print()
report_header()
out_float = test_report("float32", cnn_float_tflite, w_test, y_test)
if task_4_complete:
    out_int8 = test_report("int8, all classes", cnn_int8_tflite, w_test, y_test)
    print()
    print("The accuracy and the clipped input values are in percent.")
    print(f"largest difference between the float output and the int8 output: "
          f"{np.abs(out_float - out_int8).max():.4f}")
    print(f"windows with a different class: "
          f"{int(np.sum(out_float.argmax(axis=1) != out_int8.argmax(axis=1)))}"
          f" of {len(w_test)}")
else:
    print("Task 4 is not complete. The table has no int8 model.")

Write the two rows of the table in `report.md`. Use the rule of the lecture
for the next step:

- a loss below 1 point: use the model,
- a loss of 1 to 3 points: try quantization-aware training,
- a loss above 3 points: find the layer that causes the error.

## 8. Look into the `int8` model (8 min)

The next cell prints each tensor of the `int8` model that has a scale. Look
for three facts of the lecture:

1. The input tensor has the scale and the zero point of the range of the
   calibration set. Compare them with the result of your function
   `scale_zero_point`.
2. A weight tensor of a convolution has one scale for each filter, and the
   zero point 0.
3. A bias has the type `int32`.

In [ ]:
if task_4_complete:
    interpreter = make_interpreter(cnn_int8_tflite)
    operators = [op["op_name"] for op in interpreter._get_ops_details()
                 if op["op_name"] != "DELEGATE"]
    print("operators:", operators)
    print()
    print(f"{'tensor':<8}{'shape':<16}{'type':<8}{'scales':>7}"
          f"{'first scale':>14}{'zero point':>12}")
    for detail in interpreter.get_tensor_details():
        parameters = detail["quantization_parameters"]
        if len(parameters["scales"]) == 0:
            continue
        shape = "x".join(str(int(v)) for v in detail["shape"])
        print(f"{detail['index']:<8}{shape:<16}{np.dtype(detail['dtype']).name:<8}"
              f"{len(parameters['scales']):>7}{parameters['scales'][0]:>14.6f}"
              f"{int(parameters['zero_points'][0]):>12}")

    inp = interpreter.get_input_details()[0]
    S_in, Z_in = inp["quantization"]
    S_own, Z_own = scale_zero_point(float(calibration.min()),
                                    float(calibration.max()))
    print()
    print(f"input tensor of the model: S = {S_in:.6f}, Z = {Z_in}")
    print(f"your function for the range of the calibration set: "
          f"S = {S_own:.6f}, Z = {Z_own}")
else:
    print("Task 4 is not complete.")

The converter changes each convolution along the time axis into a
`CONV_2D` operator with a height of 1. For this reason, the list has the
operators `EXPAND_DIMS` and `RESHAPE`.

## 9. Change the calibration set (17 min)

The conversion gives no error when the calibration set is wrong. You now
make this error on purpose.

Experiment 1 uses a calibration set with **one class only**, for each class.
Experiment 2 uses a calibration set with all classes and a smaller number of
windows.

**Task 6.** Write your prediction **before** you run the experiment. The
table of section 1 gives the range of each class.

In [ ]:
# TODO (student): replace each "?".
prediction_calibration = {
    # a class name
    "the calibration class with the lowest accuracy": "?",
    # a class name, or "none"
    "a calibration class with no loss of accuracy": "?",
    # a number of windows: 4, 20, or 200
    "the smallest set of all classes with no loss": "?",
    # one sentence
    "the reason": "?",
}
for question, answer in prediction_calibration.items():
    print(f"{question:<50} {answer}")

In [ ]:
if task_4_complete:
    print("Experiment 1: 50 calibration windows of one class")
    report_header()
    test_report("float32", cnn_float_tflite, w_test, y_test)
    test_report("int8, all classes", cnn_int8_tflite, w_test, y_test)
    for number, label in enumerate(labels):
        one_class = calibration_set(w_train, y_train, each=50, classes=[number])
        model_one = convert_int8(cnn, one_class)
        test_report(f"int8, only {label[:11]}", model_one, w_test, y_test)
    print()
    print("Experiment 2: all classes, a smaller calibration set")
    report_header()
    for each in (50, 5, 1):
        small = calibration_set(w_train, y_train, each=each)
        model_small = convert_int8(cnn, small)
        test_report(f"int8, {len(small)} windows", model_small, w_test, y_test)
else:
    print("Task 4 is not complete.")

> TODO (student): answer these three questions in `report.md`.
>
> 1. Compare the result with your prediction of task 6.
> 2. Which column of the table shows the problem when you have no labels for
>    the test inputs?
> 3. Your product gets a new motion class next year. Which two steps do you
>    repeat for the `int8` model?

# Part C: quantization-aware training (40 min)

## 10. Simulated quantization (10 min)

Quantization-aware training keeps the weights in `float32`. In the forward
pass, each weight and each activation goes through the mapping: quantize,
then dequantize. The result is a float value with the error of an integer.

The function `fake_quant` does this for a tensor with the range
$[x_{\min}, x_{\max}]$. It uses the integers from 0 to $2^b - 1$. For 8
bits, this is the `int8` mapping with all integers moved by 128. The real
values are the same.

The round function has a slope of 0, so the training gets no gradient. The
straight-through estimator of the lecture repairs this: the forward pass
uses the quantized value `x_q`, and the backward pass uses the identity.

**Task 7.** Replace the last line of `fake_quant` with the straight-through
estimator. Use `ops.stop_gradient`.

In [ ]:
def fake_quant(x, x_min, x_max, bits=8):
    '''Quantize and dequantize x in the forward pass.'''
    levels = 2.0 ** bits - 1.0
    S = ops.maximum(x_max - x_min, 1e-8) / levels
    Z = ops.round(-x_min / S)
    q = ops.clip(ops.round(x / S) + Z, 0.0, levels)
    x_q = S * (q - Z)
    # TODO (student): replace the line below. The forward value must be x_q.
    # The gradient must be the gradient of x.
    return x_q

In [ ]:
def check_task_7():
    '''Check the forward value and the gradient of fake_quant.'''
    values = np.array([-0.5, 0.0, 0.3, 0.7, 1.7, 2.9], dtype=np.float32)
    x = tf.Variable(values)
    with tf.GradientTape() as tape:
        y = fake_quant(x, -1.0, 3.0)
        total = ops.sum(y)
    gradient = tape.gradient(total, x)
    S, Z = 4.0 / 255, -64
    expected = S * (np.clip(np.round(values / S) + Z, -128, 127) - Z)
    forward = np.allclose(np.array(y), expected, atol=1e-5)
    backward = gradient is not None and np.allclose(np.array(gradient), 1.0)
    print("forward value:", np.round(np.array(y), 5))
    print("gradient:     ", None if gradient is None else np.array(gradient))
    print("Task 7:", "complete" if forward and backward else "not complete")
    return forward and backward


task_7_complete = check_task_7()

The next cell has the layers of the model with simulated quantization. Read
them. You do not change this code.

- `fake_quant_weights` quantizes a weight tensor with a symmetric range and
  one scale for each output channel, as the converter does. With $b$ bits, a
  weight has $2^b - 1$ values. With 2 bits, the values are $-S$, 0, and
  $+S$: a ternary weight.
- `Observer` quantizes an activation. It keeps the range of the activation.
  In the calibration it records the minimum and the maximum. In the training
  it keeps a moving average.
- `QuantConv1D` and `QuantDense` run a layer of the float model with
  quantized weights. The float weights stay the variables of the training.

In [ ]:
def fake_quant_weights(w, bits):
    '''Fake quantization of weights: symmetric, one scale for each channel.'''
    top = 2.0 ** (bits - 1) - 1.0
    axes = list(range(len(w.shape) - 1))          # the last axis is the channel
    largest = ops.maximum(ops.max(ops.abs(w), axis=axes, keepdims=True), 1e-8)
    S = largest / top
    w_q = S * ops.clip(ops.round(w / S), -top, top)
    return w + ops.stop_gradient(w_q - w)


class Observer(keras.layers.Layer):
    '''Fake quantization of an activation with a range that the layer records.'''

    def __init__(self, bits, momentum=0.99, **kwargs):
        super().__init__(**kwargs)
        self.bits = bits
        self.momentum = momentum
        self.calibrate = False

    def build(self, input_shape):
        self.r_min = self.add_weight(shape=(), initializer="zeros",
                                     trainable=False, name="r_min")
        self.r_max = self.add_weight(shape=(), initializer="ones",
                                     trainable=False, name="r_max")

    def call(self, x, training=False):
        if self.calibrate:                        # record, and change nothing
            self.r_min.assign(ops.minimum(self.r_min, ops.min(x)))
            self.r_max.assign(ops.maximum(self.r_max, ops.max(x)))
            return x
        if self.bits is None:                     # the activations stay float
            return x
        if training:                              # moving average of the range
            keep = self.momentum
            self.r_min.assign(keep * self.r_min
                              + (1 - keep) * ops.minimum(ops.min(x), 0.0))
            self.r_max.assign(keep * self.r_max
                              + (1 - keep) * ops.maximum(ops.max(x), 0.0))
        return fake_quant(x, self.r_min, self.r_max, self.bits)


class QuantConv1D(keras.layers.Layer):
    '''A convolution of the float model with fake quantization of the weights.'''

    def __init__(self, source, bits, **kwargs):
        super().__init__(**kwargs)
        self.source = source
        self.bits = bits

    def call(self, x):
        w = self.source.kernel
        if self.bits is not None:
            w = fake_quant_weights(w, self.bits)
        y = ops.conv(x, w, strides=self.source.strides, padding="valid")
        return ops.relu(y + self.source.bias)


class QuantDense(keras.layers.Layer):
    '''A fully connected layer with fake quantization of the weights.'''

    def __init__(self, source, bits, **kwargs):
        super().__init__(**kwargs)
        self.source = source
        self.bits = bits

    def call(self, x):
        w = self.source.kernel
        if self.bits is not None:
            w = fake_quant_weights(w, self.bits)
        return ops.matmul(x, w) + self.source.bias


LAYERS = ["convolution 1", "convolution 2", "fully connected"]


def build_quant_model(float_model, weight_bits, activation_bits=8):
    '''Return (model with simulated quantization, float copy of the model).

    weight_bits: one number for all layers, or a list with one entry for each
    layer of LAYERS. The entry None keeps the weights of a layer in float.
    activation_bits: the bits of all activations. None keeps them in float.
    The two models use the same weight variables.
    '''
    if not isinstance(weight_bits, (list, tuple)):
        weight_bits = [weight_bits] * len(LAYERS)
    base = build_cnn()
    base.set_weights(float_model.get_weights())
    conv_1, conv_2, pooling, dense = base.layers
    inputs = keras.Input(shape=(WINDOW, 3))
    x = Observer(activation_bits)(inputs)
    x = Observer(activation_bits)(QuantConv1D(conv_1, weight_bits[0])(x))
    x = Observer(activation_bits)(QuantConv1D(conv_2, weight_bits[1])(x))
    x = Observer(activation_bits)(pooling(x))
    x = Observer(activation_bits)(QuantDense(dense, weight_bits[2])(x))
    outputs = keras.layers.Softmax()(x)
    return keras.Model(inputs, outputs), base


def calibrate(quant_model, x):
    '''Record the range of each activation for the inputs x.'''
    observers = [layer for layer in quant_model.layers
                 if isinstance(layer, Observer)]
    for observer in observers:
        observer.calibrate = True
        observer.r_min.assign(0.0)
        observer.r_max.assign(0.0)
    quant_model(x, training=False)
    for observer in observers:
        observer.calibrate = False


def accuracy_of(model, x, y):
    '''Return the accuracy of a Keras model in percent.'''
    return class_accuracy(model.predict(x, verbose=0).argmax(axis=1), y)[0]


quant_8, _ = build_quant_model(cnn, 8)
calibrate(quant_8, calibration)
print(f"float model:                       {accuracy_of(cnn, w_test, y_test):.1f} percent")
print(f"simulated quantization, 8 bits:    {accuracy_of(quant_8, w_test, y_test):.1f} percent")
if task_4_complete:
    print(f"int8 LiteRT model of section 7:    "
          f"{class_accuracy(out_int8.argmax(axis=1), y_test)[0]:.1f} percent")
print()
print("ranges that the observers recorded:")
for layer, name in zip([l for l in quant_8.layers if isinstance(l, Observer)],
                       ["input", "after convolution 1", "after convolution 2",
                        "after the pooling", "before the softmax"]):
    print(f"  {name:<22}{float(layer.r_min):>8.3f} to {float(layer.r_max):>7.3f}")

The model with simulated quantization and the `int8` LiteRT model must give
almost the same accuracy. The simulation is a model of the integer kernels.
It is not a copy of them.

## 11. Train with simulated quantization (12 min)

With 8 bits, the model loses almost nothing. Quantization-aware training
then has nothing to repair. To see its effect, the experiment also uses
weights with 4, 3, and 2 bits. The activations keep 8 bits.

For each number of bits, the code measures two models:

- **no new training:** the float weights with simulated quantization. This
  is post-training quantization.
- **quantization-aware training:** the same model after 10 epochs of training
  with simulated quantization. The learning rate is a tenth of the rate of
  the float training.

LiteRT has no format for weights with fewer than 8 bits. The rows for 4, 3,
and 2 bits are a simulation only.

**Task 8.** Write your prediction **before** you run the experiment.

In [ ]:
# TODO (student): replace each "?".
prediction_bits = {
    # a number in percent
    "accuracy with weights of 2 bits and no new training": "?",
    # "yes" or "no"
    "quantization-aware training repairs the loss at 2 bits": "?",
    # one sentence
    "the reason": "?",
}
for question, answer in prediction_bits.items():
    print(f"{question:<56} {answer}")

In [ ]:
def quantization_aware_training(float_model, weight_bits, epochs=10):
    '''Return (accuracy with no new training, accuracy after the training,
    float copy of the model with the trained weights).'''
    keras.utils.set_random_seed(0)
    quant_model, base = build_quant_model(float_model, weight_bits)
    calibrate(quant_model, calibration)
    before = accuracy_of(quant_model, w_test, y_test)
    compile_model(quant_model, 0.0005)
    quant_model.fit(w_train, y_train, epochs=epochs, batch_size=32, verbose=0)
    after = accuracy_of(quant_model, w_test, y_test)
    return before, after, base


qat_models = {}
if task_7_complete:
    print(f"float model: {float_accuracy:.1f} percent")
    print()
    print(f"{'weight bits':>11}{'values':>8}{'no new training':>17}"
          f"{'quantization-aware training':>29}")
    for bits in (8, 4, 3, 2):
        before, after, qat_models[bits] = quantization_aware_training(cnn, bits)
        print(f"{bits:>11}{2 ** bits - 1:>8}{before:>16.1f}%{after:>28.1f}%")
else:
    print("Task 7 is not complete. The training needs the straight-through")
    print("estimator.")

> TODO (student): answer these two questions in `report.md`.
>
> 1. Compare the result with your prediction of task 8.
> 2. The board of this course runs `int8` kernels only. Is quantization-aware
>    training necessary for this model? Give your numbers.

## 12. Find the layer that loses accuracy (8 min)

Part 3 of the lecture gave a method: quantize one layer at a time and
measure the accuracy. The next cell does this with no new training. One
layer has weights with a small number of bits. The weights of the other
layers and all activations stay in float.

In [ ]:
print(f"float model: {float_accuracy:.1f} percent")
layer_parameters = [cnn.layers[0].count_params(), cnn.layers[1].count_params(),
                    cnn.layers[3].count_params()]
for low_bits in (4, 2):
    print()
    print(f"{'layer with weights of ' + str(low_bits) + ' bits':<32}"
          f"{'parameters':>12}{'accuracy':>10}{'loss':>8}")
    layer_loss = []
    for number, layer in enumerate(LAYERS + ["all layers"]):
        weight_bits = [None] * len(LAYERS)
        if number < len(LAYERS):
            weight_bits[number] = low_bits
            parameters = layer_parameters[number]
        else:
            weight_bits = low_bits
            parameters = sum(layer_parameters)
        model_one, _ = build_quant_model(cnn, weight_bits, activation_bits=None)
        accuracy = accuracy_of(model_one, w_test, y_test)
        layer_loss.append(float_accuracy - accuracy)
        print(f"{layer:<32}{parameters:>12}{accuracy:>9.1f}%"
              f"{float_accuracy - accuracy:>8.1f}")
    print(f"sum of the losses of the single layers: {sum(layer_loss[:-1]):.1f}")

> TODO (student): answer these two questions in `report.md`.
>
> 1. For each table: which layer causes the loss? Is the sum of the single
>    losses near the loss of the complete model?
> 2. All weights must have 4 bits, and one layer can keep 8 bits. Which
>    layer do you select? Give the cost in bytes: a weight with 8 bits needs
>    1 byte, and a weight with 4 bits needs half a byte.

## 13. From the trained model to a real `int8` model (10 min)

The model with simulated quantization is a float model. The board needs a
real `int8` file. The steps of the lecture:

1. copy the trained weights into the float model,
2. convert with the same settings as in Part B,
3. test the real `int8` model.

The next cell does this for the model that trained with 8 bits. It uses your
function `convert_int8`.

In [ ]:
if task_4_complete and task_7_complete:
    cnn_qat_tflite = convert_int8(qat_models[8], calibration)
    report_header()
    test_report("float32", cnn_float_tflite, w_test, y_test)
    out_ptq = test_report("int8, post-training", cnn_int8_tflite, w_test, y_test)
    out_qat = test_report("int8, aware training", cnn_qat_tflite, w_test, y_test)
    operators = [[op["op_name"] for op in make_interpreter(m)._get_ops_details()
                  if op["op_name"] != "DELEGATE"]
                 for m in (cnn_int8_tflite, cnn_qat_tflite)]
    print()
    print("the two int8 files have the same operators:", operators[0] == operators[1])
    print(f"windows with a different class in the two int8 models: "
          f"{int(np.sum(out_ptq.argmax(axis=1) != out_qat.argmax(axis=1)))}"
          f" of {len(w_test)}")
else:
    print("Tasks 4 and 7 are not complete.")

Write the three rows in `report.md`. The two `int8` files have the same
format and the same operators. The board sees no difference between them.
Only the values of the weights are different.

# Part D: the float model and the `int8` model for the board (30 min)

## 14. The feature model of Day 3 in two formats (3 min)

The board runs the model of Day 3: 63 features, two hidden layers with 20
and 10 neurons, and 4 outputs. The code of this section is the code of the
Day 3 notebook: the features, the normalization, and the training.

The code then converts the model two times: to `float32`, and to `int8` with
your function `convert_int8`. The calibration set holds normalized features
of the training set.

You do not change this code.

In [ ]:
def moments(x):
    '''Return (skewness, kurtosis) of the values. 0 for a constant signal.'''
    d = x - x.mean()
    m2 = np.mean(d ** 2)
    if m2 < 1e-12:
        return 0.0, 0.0
    return np.mean(d ** 3) / m2 ** 1.5, np.mean(d ** 4) / m2 ** 2 - 3.0


def spectral_power(x, nfft=FFT_LENGTH):
    '''Largest power of each frequency bin over the frames of the window.'''
    power = np.zeros(nfft // 2 + 1)
    for start in range(0, len(x), nfft):
        frame = x[start:start + nfft]                  # the last frame is short
        spectrum = np.abs(np.fft.rfft(frame, nfft))    # rfft adds zeros
        power = np.maximum(power, spectrum ** 2 / nfft)
    return power[1:]                                   # no bin at 0 Hz


def axis_features(x):
    '''Return the 21 features of one axis.'''
    centred = x - x.mean()
    rms_value = np.sqrt(np.mean(centred ** 2))
    skewness, kurtosis = moments(centred)
    power = spectral_power(centred)
    power_skewness, power_kurtosis = moments(power)
    return np.concatenate([[rms_value, skewness, kurtosis, power_skewness,
                            power_kurtosis], power])


def extract_features(window):
    '''Return the 63 features of one window with the shape (100, 3).'''
    return np.concatenate([axis_features(window[:, k])
                           for k in range(window.shape[1])]).astype(np.float32)


f_train = np.array([extract_features(w) for w in w_train])
f_test = np.array([extract_features(w) for w in w_test])
N_FEATURES = f_train.shape[1]
feature_mean = f_train.mean(axis=0)
feature_std = np.maximum(f_train.std(axis=0), 1e-6)
x_train = (f_train - feature_mean) / feature_std
x_test = (f_test - feature_mean) / feature_std

keras.utils.set_random_seed(0)
feature_model = keras.Sequential([
    keras.Input(shape=(N_FEATURES,), name="keras_tensor"),
    keras.layers.Dense(20, activation="relu", name="dense"),
    keras.layers.Dense(10, activation="relu", name="dense_1"),
    keras.layers.Dense(len(labels), activation="softmax", name="dense_2"),
], name="sequential")
compile_model(feature_model, 0.005)
feature_model.fit(x_train, y_train, epochs=30, batch_size=32, verbose=0)

board_float_tflite = convert_quietly(
    tf.lite.TFLiteConverter.from_keras_model(feature_model))
feature_calibration = calibration_set(x_train, y_train, each=50)
board_int8_tflite = convert_int8(feature_model, feature_calibration)
board_ready = (task_4_complete and task_5_complete
               and input_details_of(board_int8_tflite)["dtype"] == np.int8)

print(f"feature model: {feature_model.count_params()} parameters")
print()
report_header()
board_out_float = test_report("float32", board_float_tflite, x_test, y_test)
if board_ready:
    board_out_int8 = test_report("int8", board_int8_tflite, x_test, y_test)
    details = input_details_of(board_int8_tflite)
    print()
    print(f"input tensor of the int8 model: S = {details['quantization'][0]:.6f},"
          f" Z = {details['quantization'][1]}")
    print(f"smaller file: {len(board_float_tflite) - len(board_int8_tflite)} bytes, "
          f"{100 * len(board_int8_tflite) / len(board_float_tflite):.0f} percent "
          "of the float file")
else:
    print("Tasks 4 and 5 are not complete. The notebook has no int8 model.")

The next cell gives the size of each activation tensor of the two models.
Use the rule of Day 3: the peak is the largest sum of the input and the
output of one operator. Write the two peaks in `report.md`. You compare them
with the arena of the board.

In [ ]:
def activation_table(tflite_model):
    '''Print the input size and the output size of each operator.'''
    interpreter = make_interpreter(tflite_model)
    details = {d["index"]: d for d in interpreter.get_tensor_details()}

    def size(index):
        return (int(np.prod(details[index]["shape"]))
                * np.dtype(details[index]["dtype"]).itemsize)

    peak = 0
    print(f"  {'operator':<18}{'input bytes':>12}{'output bytes':>14}{'sum':>8}")
    for op in interpreter._get_ops_details():
        if op["op_name"] == "DELEGATE":
            continue
        first_input, output = size(op["inputs"][0]), size(op["outputs"][0])
        peak = max(peak, first_input + output)
        print(f"  {op['op_name']:<18}{first_input:>12}{output:>14}"
              f"{first_input + output:>8}")
    print(f"  peak of the activations: {peak} bytes")


print("float32 model")
activation_table(board_float_tflite)
if board_ready:
    print()
    print("int8 model")
    activation_table(board_int8_tflite)

## 15. Write the files for the sketch (2 min)

The next cell writes four files into `sketches/motion_quant/`:

| File | Content |
|---|---|
| `model_float.h` | The `float32` LiteRT file as a C array |
| `model_int8.h` | The `int8` LiteRT file as a C array |
| `model_settings.h` | The class names, and the mean and the standard deviation of each feature |
| `test_set.h` | The features of some test windows, the real class of each window, and the class that each model gives on the laptop |

The sketch runs the test set on the board. It prints the accuracy and the
time of each model, and it compares each class with the class of the laptop.

In [ ]:
TEST_WINDOWS_EACH = 20       # test windows of each class for the board


def c_float(value):
    '''Return a number as a C float constant, for example 1.0f or 2.5e-07f.'''
    text = f"{float(value):.9g}"
    if "." not in text and "e" not in text:
        text += ".0"
    return text + "f"


def c_array(name, values, per_line=6, kind="float"):
    '''Return the C text of a constant array.'''
    lines = [f"const {kind} {name}[{len(values)}] = {{"]
    for start in range(0, len(values), per_line):
        chunk = values[start:start + per_line]
        if kind == "float":
            lines.append("    " + ", ".join(c_float(v) for v in chunk) + ",")
        else:
            lines.append("    " + ", ".join(str(int(v)) for v in chunk) + ",")
    lines.append("};")
    return "\n".join(lines)


def board_test_positions():
    '''Return the positions of the test windows for the board.'''
    positions = []
    for k in range(len(labels)):
        of_class = np.flatnonzero(y_test == k)
        if len(of_class) == 0:
            continue
        step = max(1, len(of_class) // TEST_WINDOWS_EACH)
        positions.extend(of_class[::step][:TEST_WINDOWS_EACH])
    return np.array(positions)


def write_sketch_files():
    '''Write the four files. Return (positions of the board test set,
    number of clipped input values of the int8 model for this set).'''
    import tflite_to_header                 # Labs/day04/host/

    os.makedirs(SKETCH, exist_ok=True)
    note = ("Made by quantization.ipynb. Do not change this file by hand.\n"
            "// Dataset: " + os.path.basename(os.path.normpath(DATA)))

    for name, data, guard in (
            ("model_float.h", board_float_tflite, "MODEL_FLOAT_H_"),
            ("model_int8.h", board_int8_tflite, "MODEL_INT8_H_")):
        text = tflite_to_header.make_header(data, name.replace(".h", ".tflite"))
        with open(os.path.join(SKETCH, name), "w", encoding="ascii") as handle:
            handle.write(text.replace("MODEL_H_", guard))

    names = ", ".join(f'"{label}"' for label in labels)
    with open(os.path.join(SKETCH, "model_settings.h"), "w",
              encoding="ascii") as handle:
        handle.write(f"// model_settings.h - settings of the motion classifier\n"
                     f"// {note}\n\n"
                     "#ifndef MODEL_SETTINGS_H_\n#define MODEL_SETTINGS_H_\n\n"
                     f"constexpr int kNumClasses = {len(labels)};\n"
                     f"const char* const kClassNames[kNumClasses] = {{{names}}};\n\n"
                     "// Mean and standard deviation of each feature in the "
                     "training set.\n"
                     + c_array("kFeatureMean", feature_mean) + "\n\n"
                     + c_array("kFeatureStd", feature_std) + "\n\n"
                     "#endif  // MODEL_SETTINGS_H_\n")

    positions = board_test_positions()
    S_in, Z_in = input_details_of(board_int8_tflite)["quantization"]
    _, clipped = quantize_input(x_test[positions], S_in, Z_in)
    with open(os.path.join(SKETCH, "test_set.h"), "w", encoding="ascii") as handle:
        handle.write("// test_set.h - test windows with their known results\n"
                     f"// {note}\n\n"
                     "#ifndef TEST_SET_H_\n#define TEST_SET_H_\n\n"
                     "#include <stdint.h>\n\n"
                     f"constexpr int kTestCount = {len(positions)};\n\n"
                     "// Input values of the int8 model that the clamp changes "
                     "on the laptop.\n"
                     f"constexpr int kTestInt8Clipped = {int(clipped)};\n\n"
                     "// The 63 features of each test window, before the "
                     "normalization:\n// window 0, window 1, ...\n"
                     + c_array("kTestFeatures", f_test[positions].reshape(-1))
                     + "\n\n// The real class of each window.\n"
                     + c_array("kTestLabel", y_test[positions], 20, "uint8_t")
                     + "\n\n// The class of the float32 model on the laptop.\n"
                     + c_array("kTestFloatClass",
                               board_out_float[positions].argmax(axis=1), 20,
                               "uint8_t")
                     + "\n\n// The class of the int8 model on the laptop.\n"
                     + c_array("kTestInt8Class",
                               board_out_int8[positions].argmax(axis=1), 20,
                               "uint8_t")
                     + "\n\n#endif  // TEST_SET_H_\n")
    return positions, int(clipped)


if board_ready:
    positions, board_clipped = write_sketch_files()
    for name in ("model_float.h", "model_int8.h", "model_settings.h",
                 "test_set.h"):
        path = os.path.join(SKETCH, name)
        print(f"wrote {os.path.normpath(path)} ({os.path.getsize(path)} bytes)")
    print()
    print(f"test set for the board: {len(positions)} windows")
    for name, outputs in (("float32", board_out_float), ("int8", board_out_int8)):
        correct = int(np.sum(outputs[positions].argmax(axis=1) == y_test[positions]))
        print(f"  {name:<8} correct on the laptop: {correct} of {len(positions)} "
              f"({100 * correct / len(positions):.1f} percent)")
    print(f"  int8     clipped input values on the laptop: {board_clipped} of "
          f"{len(positions) * N_FEATURES}")
else:
    print("Tasks 4 and 5 are not complete. The notebook did not write the files.")

## Summary

- Two numbers describe the `int8` mapping of a tensor: the scale and the
  zero point. The range of the tensor decides them.
- Post-training quantization needs a calibration set from all classes. A
  wrong calibration set gives a wrong model with no error message. The
  number of clipped input values shows the problem.
- Quantization-aware training repairs a loss of accuracy. With 8 bits, this
  small model has no loss to repair.
- Continue with Part D of the lab: open `sketches/motion_quant/` in the
  Arduino IDE.